In [1]:
pip install tensorflow opencv-python scikit-learn matplotlib


In [3]:
import zipfile
import os
import random
import shutil
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# ==============================
# PATHS
# ==============================
ZIP_PATH = "Bill_dataset.zip"
DATASET_DIR = "dataset"
TRAIN_DIR = "dataset/train"
TEST_DIR = "dataset/test"
IMG_SIZE = 128
TEST_IMAGES_PER_CLASS = 5



In [4]:
if not os.path.exists(DATASET_DIR):
    os.makedirs(DATASET_DIR)

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(DATASET_DIR)

print("✅ Dataset Extracted")

✅ Dataset Extracted


In [5]:
os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

classes = os.listdir(os.path.join(DATASET_DIR, "Bill_dataset"))

for cls in classes:
    os.makedirs(os.path.join(TRAIN_DIR, cls), exist_ok=True)
    os.makedirs(os.path.join(TEST_DIR, cls), exist_ok=True)

print("✅ Train & Test folders created")

✅ Train & Test folders created


In [7]:
for cls in classes:
    cls_path = os.path.join(DATASET_DIR, "Bill_dataset", cls)
    # Skip if it's not a directory (e.g., .DS_Store file)
    if not os.path.isdir(cls_path):
        continue

    images = os.listdir(cls_path)
    random.shuffle(images)

    test_images = images[:TEST_IMAGES_PER_CLASS]
    train_images = images[TEST_IMAGES_PER_CLASS:]

    for img in test_images:
        shutil.move(
            os.path.join(cls_path, img),
            os.path.join(TEST_DIR, cls, img)
        )

    for img in train_images:
        shutil.move(
            os.path.join(cls_path, img),
            os.path.join(TRAIN_DIR, cls, img)
        )

print("✅ Images split successfully (NO overlap)")

✅ Images split successfully (NO overlap)


In [9]:
def load_data(folder):
    X, y = [], []
    # Get class names by listing directories in the given folder
    # Filter out any non-directory entries like .DS_Store if they exist at this level
    class_names = sorted([d for d in os.listdir(folder) if os.path.isdir(os.path.join(folder, d))])

    for idx, cls in enumerate(class_names):
        cls_folder = os.path.join(folder, cls)
        for img_name in os.listdir(cls_folder):
            img_path = os.path.join(cls_folder, img_name)
            # Ensure it's a file and not a directory (e.g., .DS_Store)
            if not os.path.isfile(img_path):
                continue

            img = cv2.imread(img_path)
            # Check if image was loaded successfully
            if img is None:
                print(f"Warning: Could not load image {img_path}. Skipping.")
                continue

            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            X.append(img)
            y.append(idx)

    return np.array(X) / 255.0, np.array(y), class_names

X_train, y_train, class_names = load_data(TRAIN_DIR)
X_test, y_test, _ = load_data(TEST_DIR)

y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

print("✅ Data Loaded")

✅ Data Loaded


In [10]:
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128, activation='relu'),
    Dense(len(class_names), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("✅ Model Compiled")

✅ Model Compiled


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [11]:
model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=16,
    validation_split=0.2
)

Epoch 1/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 250ms/step - accuracy: 0.4687 - loss: 2.9750 - val_accuracy: 0.0000e+00 - val_loss: 2.9712
Epoch 2/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 220ms/step - accuracy: 0.5933 - loss: 0.8311 - val_accuracy: 0.0000e+00 - val_loss: 2.3433
Epoch 3/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 244ms/step - accuracy: 0.8336 - loss: 0.4459 - val_accuracy: 0.0000e+00 - val_loss: 3.4197
Epoch 4/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 233ms/step - accuracy: 0.9184 - loss: 0.2961 - val_accuracy: 0.6897 - val_loss: 2.0150
Epoch 5/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 223ms/step - accuracy: 0.9662 - loss: 0.0898 - val_accuracy: 0.6552 - val_loss: 2.2545
Epoch 6/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 305ms/step - accuracy: 1.0000 - loss: 0.0399 - val_accuracy: 0.7241 - val_loss: 2.3486
Epoch 7/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 225ms/step - accuracy: 1.0000 - loss: 0.0160 - val_accuracy: 0.7241 - val_loss: 3.2462
Epoch 8/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 268ms/step - accuracy: 1.0000 - loss: 0.0038 - val_accuracy: 0.6897

In [12]:
test_loss, test_accuracy = model.evaluate(X_test, y_test)

print("🎯 Test Accuracy:", round(test_accuracy * 100, 2), "%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step - accuracy: 0.7500 - loss: 3.4884
🎯 Test Accuracy: 75.0 %
